In [17]:
import pandas as pd
import numpy as np
import os

silver_path = "../data/silver/"
gold_path = "../data/gold/"
os.makedirs(gold_path, exist_ok=True)

# Cargar datos de la capa Silver
df_clientes = pd.read_parquet(f"{silver_path}clientes.parquet")
df_estado = pd.read_parquet(f"{silver_path}cliente_estado_mensual.parquet")
df_prod_mensual = pd.read_parquet(f"{silver_path}cliente_producto_mensual.parquet")
df_altas = pd.read_parquet(f"{silver_path}cliente_producto_alta.parquet")
df_productos = pd.read_parquet(f"{silver_path}productos.parquet")

print(" Datos de la capa Silver cargados correctamente.")

 Datos de la capa Silver cargados correctamente.


In [18]:
print("Generando matriz de productos candidatos...")

# 1. Obtener todas las combinaciones únicas de cliente y mes presentes en el historial médico/financiero
base_cliente_mes = df_prod_mensual[['id_cliente', 'fecha_corte']].drop_duplicates()

# 2. Hacer un cross join (producto cartesiano) con el catálogo de productos para tener todos los candidatos posibles
todos_los_productos = df_productos[['id_producto']].drop_duplicates()
base_cliente_mes['key'] = 1
todos_los_productos['key'] = 1

dataset_base = pd.merge(base_cliente_mes, todos_los_productos, on='key').drop(columns=['key'])

# 3. Filtrar y EXCLUIR los productos que el cliente YA TIENE activos en ese mes (Regla 15.3)
dataset_base = pd.merge(
    dataset_base, 
    df_prod_mensual[['id_cliente', 'fecha_corte', 'id_producto', 'estado_producto']], 
    on=['id_cliente', 'fecha_corte', 'id_producto'], 
    how='left'
)

# Nos quedamos solo con los productos que NO tiene activos (estado_producto == 0 o NaN)
dataset_final = dataset_base[dataset_base['estado_producto'].fillna(0) == 0].drop(columns=['edges', 'estado_producto'], errors='ignore')

print(f" Matriz base construida. El total de filas a evaluar es: {len(dataset_final)}")

Generando matriz de productos candidatos...
 Matriz base construida. El total de filas a evaluar es: 14947


In [19]:
print("Calculando la variable objetivo (Target 'y')...")

# 1. Calcular la fecha del próximo mes (t + 1) para cruzar con las altas
# Nota: Sumamos un mes aproximado usando DateOffset
df_altas['fecha_observacion_previa'] = df_altas['fecha_corte'] - pd.DateOffset(months=1)

# 2. Crear una columna auxiliar en altas para marcar que sí hubo compra (y = 1)
df_altas['y'] = 1

# 3. Cruzar nuestra matriz base con la de altas usando (mes t) con (mes t+1 de alta)
dataset_final = pd.merge(
    dataset_final,
    df_altas[['id_cliente', 'id_producto', 'fecha_observacion_previa', 'y']],
    left_on=['id_cliente', 'id_producto', 'fecha_corte'],
    right_on=['id_cliente', 'id_producto', 'fecha_observacion_previa'],
    how='left'
)

# 4. Los que no cruzaron significa que no compraron el producto en t+1, por ende y = 0
dataset_final['y'] = dataset_final['y'].fillna(0).astype(int)
dataset_final = dataset_final.drop(columns=['fecha_observacion_previa'])

print(f" Distribución del target 'y':\n{dataset_final['y'].value_counts()}")

Calculando la variable objetivo (Target 'y')...
 Distribución del target 'y':
y
0    14947
Name: count, dtype: int64


In [20]:
print("Agregando características (Features) al dataset...")

# Forzamos a que las fechas de ambas tablas sean tratadas exactamente igual (como strings limpios)
dataset_final['fecha_corte_str'] = pd.to_datetime(dataset_final['fecha_corte']).dt.strftime('%Y-%m-%d')
df_estado['fecha_corte_str'] = pd.to_datetime(df_estado['fecha_corte']).dt.strftime('%Y-%m-%d')

# Asegurar que los IDs de cliente sean del mismo tipo numérico
dataset_final['id_cliente'] = dataset_final['id_cliente'].astype(int)
df_estado['id_cliente'] = df_estado['id_cliente'].astype(int)


# 1. Cruzar con la tabla maestra de clientes
dataset_final = pd.merge(dataset_final, df_clientes, on='id_cliente', how='inner')

# 2. Cruzar con el estado mensual usando la columna corregida de texto
dataset_final = pd.merge(dataset_final, df_estado.drop(columns=['fecha_corte']), on=['id_cliente', 'fecha_corte_str'], how='inner')
dataset_final = dataset_final.drop(columns=['fecha_corte_str'])

# 3. Cruzar con los atributos del producto candidato
dataset_final = pd.merge(
    dataset_final, 
    df_productos[['id_producto', 'nombre_producto', 'categoria_producto']], 
    on='id_producto', 
    how='inner'
)

# Renombrar id_producto a id_producto_candidato
dataset_final = dataset_final.rename(columns={
    'id_producto': 'id_producto_candidato',
    'nombre_producto': 'nombre_producto_candidato',
    'categoria_producto': 'categoria_producto_candidato'
})

print(f" El Dataset final fue enriquecido con écito . Las dimensiones reales son: {dataset_final.shape}")

Agregando características (Features) al dataset...
 El Dataset final fue enriquecido con écito . Las dimensiones reales son: (14947, 34)


In [21]:
# Definir cuál es el último mes en los datos para dejarlo como set de predicción futura
ultimo_mes = dataset_final['fecha_corte'].max()
print(f"Este es el último mes detectado para predicción en producción: {ultimo_mes.strftime('%Y-%m-%d')}")

# Dataset de predicción futura (último mes, quitamos o dejamos nulo 'y' según regla 9.6)
df_prediccion_nbp = dataset_final[dataset_final['fecha_corte'] == ultimo_mes].copy()
df_prediccion_nbp = df_prediccion_nbp.drop(columns=['y'])

# Dataset de entrenamiento (meses anteriores)
df_entrenamiento_nbp = dataset_final[dataset_final['fecha_corte'] < ultimo_mes].copy()

# Guardar en la capa Gold en formato Parquet
df_entrenamiento_nbp.to_parquet(f"{gold_path}dataset_entrenamiento_nbp.parquet", index=False)
df_prediccion_nbp.to_parquet(f"{gold_path}dataset_prediccion_nbp.parquet", index=False)

print("\n ¡La capa Gold ha sido completada exitosamente!")
print(f" Se guardo el dataset_entrenamiento_nbp.parquet con {len(df_entrenamiento_nbp)} filas.")
print(f" Se guardo el dataset_prediccion_nbp.parquet con {len(df_prediccion_nbp)} filas (Scoring).")

Este es el último mes detectado para predicción en producción: 2015-01-28

 ¡La capa Gold ha sido completada exitosamente!
 Se guardo el dataset_entrenamiento_nbp.parquet con 0 filas.
 Se guardo el dataset_prediccion_nbp.parquet con 14947 filas (Scoring).
